# 多视角金融研究 Agent · 昇腾 NPU 跑通验证

**输入一个热点，AI 自动完成投研闭环**：事件理解 → 产业链推理 → 候选股筛选 → 四因子评分 → 风险分析 → 研究报告。

本 Notebook 在 GitCode 云端环境（CPU / **NPU**）一键跑通项目核心链路，共 5 步：
1. 环境检测（Python / MindSpore / 昇腾 NPU）
2. 拉取项目源码（git clone）
3. 安装依赖
4. 昇腾加速验证（MindSpore 协方差计算）
5. 跑通一次真实金融分析（离线规则模式，30 只候选股 + 研报）

> 赛事仓库：https://gitcode.com/zhichen1024/agent_finance
> 完整网站（Streamlit UI）启动方式见文末。


In [ ]:
# ===== 1. 环境检测 =====
import sys
print("Python:", sys.version.split()[0])
try:
    import mindspore
    print("MindSpore:", mindspore.__version__)
    try:
        from mindspore import context
        context.set_context(device_target="Ascend")
        print("昇腾 NPU: 可用 (device=Ascend)")
    except Exception as e:
        print("昇腾 NPU: 检测失败，将回退 CPU:", type(e).__name__)
except ImportError:
    print("MindSpore: 未安装（第 4 步可 pip install mindspore-ascend 启用 NPU 加速）")
print("检测完成 ✔")


In [ ]:
# ===== 2. 拉取项目源码 =====
import subprocess, os
REPO = "https://gitcode.com/zhichen1024/agent_finance.git"
if not os.path.exists("agent_finance"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("agent_finance")
print("工作目录:", os.getcwd())
print("项目文件数:", len(os.listdir(".")))


In [ ]:
# ===== 3. 安装依赖 =====
# 已装可跳过（首次运行约 1-2 分钟）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("依赖安装完成 ✔")


In [ ]:
# ===== 4. 昇腾加速验证 =====
# 协方差矩阵使用昇腾原生框架 MindSpore 算子实现（ops.MatMul / ReduceMean）
# 昇腾环境自动启用 NPU 加速；无 MindSpore 时自动降级 numpy（结果一致）
import numpy as np
try:
    from src.ascend_accel import backend_info, covariance_matrix
    print("计算后端:", backend_info())
    rng = np.random.default_rng(42)
    X = rng.normal(0, 0.02, (60, 3))
    cov = covariance_matrix(X)
    print("协方差矩阵 shape:", cov.shape, "| 对角元素:", np.round(np.diag(cov), 5))
    print("昇腾加速验证通过 ✔")
except Exception as e:
    print("昇腾模块调用失败（降级 numpy 继续）:", type(e).__name__, e)


In [ ]:
# ===== 5. 跑通真实金融分析（离线规则模式，无需 LLM Key）=====
# 输入热点 → 事件解析 → 产业链推理 → 候选股筛选 → 四因子评分 → 风险 → 研报
import time
from src.pipeline import run_analysis

t0 = time.time()
r = run_analysis("低空经济", use_llm=False, enrich_market=False)
dt = time.time() - t0

stocks = r.get("stock_results", [])
report = r.get("report", "")
print(f"分析完成 | 耗时 {dt:.1f}s | 候选股票 {len(stocks)} 只 | 报告 {len(report)} 字")
print("----- 研究报告（节选）-----")
print(report[:500])


## 下一步：启动完整网站

在 Notebook 终端执行（需要 LLM Key 可配置 `.env`）：

```bash
# 终端里运行（Notebook 内可用 ! 前缀执行）
pip install -r requirements.txt
echo 'ASCEND_API_KEY=你的Key' > .env
streamlit run ui/app.py --server.port 8532 --server.headless true
```

- 侧边栏会显示 **🧠 计算后端: 昇腾 NPU**（装 mindspore-ascend 后自动启用）与 **☁️ 昇腾模型调用统计**。
- 界面截图 / 演示视频 / 性能数据见仓库 README。
- 本项目的 ReAct Agent 模式：大模型自主调用 10 个金融工具完成研究，轨迹可追溯。
